# Tetris × Laya — fine-tune on a free Colab T4

Teaches a **copy** of `laya-multilingual` to answer the two Tetris questions: how to turn the falling
piece, and which column to drop it in. Runtime → Change runtime type → **T4 GPU**, then Run all.

Upload `data/tetris_colab.zip` (built by `sh training/pack_colab.sh`) when the second cell asks.

In [ ]:
!pip -q install laya
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import files
import os, zipfile
os.makedirs('/content/work', exist_ok=True)
up = files.upload()                      # pick data/tetris_colab.zip
zipfile.ZipFile(next(iter(up))).extractall('/content/work')
%cd /content/work
!wc -l data/tetris_train.jsonl data/tetris_val.jsonl

## Base checkpoint

Downloaded from Hugging Face inside Colab, so no 650 MB upload. The copy is read only:
fine-tuning writes to `models/laya-tetris`.

In [ ]:
!hf download convaiinnovations/laya --include "multilingual/*" --local-dir models/_laya
!mv models/_laya/multilingual models/laya-base && rm -rf models/_laya
!ls -lh models/laya-base

## Fine-tune

About 40 minutes on a T4 for 2 epochs. The token embedding table (197M of the 322M parameters)
stays frozen: Tetris text uses a few hundred of the 256k tokens, and freezing it saves ~3 GB.
Pass `--train-embeddings` to unfreeze it.

In [ ]:
!python training/finetune.py --base models/laya-base --out models/laya-tetris \
    --data data/tetris_train.jsonl --val data/tetris_val.jsonl --epochs 2 --bs 32

## Check what it learned, then take the weights home

In [ ]:
!python eval/eval_headless.py --games 5 --policy laya-tetris 2>/dev/null || \
  print('eval/ is not in the zip; run the eval locally instead')

In [ ]:
!cd models && zip -qr /content/laya-tetris.zip laya-tetris && ls -lh /content/laya-tetris.zip
from google.colab import files
files.download('/content/laya-tetris.zip')     # unzip into models/ locally, then run server.py